# Multilingual Speech-to-Text Converter

This notebook builds the **speech-to-text (STT)** component for a multilingual AI assistant, using [OpenAI Whisper](https://github.com/openai/whisper).

Why Whisper for this project:
- Trained on 680k hours of multilingual audio — supports **~99 languages** out of the box.
- Can **auto-detect the spoken language** (useful since your assistant is multilingual and won't always know the input language ahead of time).
- Can transcribe in the original language, or translate directly to English.
- Runs locally, no API key required.

**Pipeline covered here:**
1. Install dependencies
2. Load a Whisper model
3. Transcribe an audio file (with language auto-detection)
4. Record audio directly from the microphone (optional, for live testing)
5. Wrap everything into a reusable `SpeechToText` class you can import into the rest of your assistant

## 1. Install dependencies

Whisper also needs `ffmpeg` installed on your system (not just via pip):
- **Ubuntu/Debian:** `sudo apt install ffmpeg`
- **macOS:** `brew install ffmpeg`
- **Windows:** `choco install ffmpeg` (or download from ffmpeg.org and add to PATH)

In [ ]:
%pip install -q openai-whisper sounddevice scipy numpy

## 2. Load a Whisper model

Model size trade-offs (speed vs. accuracy):

| Model  | Params | Relative speed | Notes |
|--------|--------|-----------------|-------|
| tiny   | 39M    | fastest          | good for quick testing |
| base   | 74M    | fast             | decent balance |
| small  | 244M   | medium           | good multilingual accuracy |
| medium | 769M   | slow             | strong accuracy |
| large  | 1.5B   | slowest          | best accuracy |

For a first working prototype, start with `"base"` or `"small"`. Switch to `"medium"`/`"large"` once the pipeline works, if you have a GPU.

In [ ]:
import whisper
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = whisper.load_model("small", device=device)

## 3. Transcribe an audio file

Put an audio file (`.wav`, `.mp3`, `.m4a`, etc.) next to this notebook, or give a full path. Whisper will auto-detect the language and transcribe it.

In [ ]:
audio_path = "sample.wav"  # <-- change to your audio file

result = model.transcribe(audio_path)

print("Detected language:", result["language"])
print("Transcript:", result["text"])

### Forcing a specific language
If you already know the language (e.g. your assistant's UI lets the user pick), pass `language=` to skip auto-detection — it's faster and slightly more accurate:

In [ ]:
result_hi = model.transcribe(audio_path, language="hi")  # e.g. "hi" = Hindi, "ne" = Nepali, "es" = Spanish
print(result_hi["text"])

### Translating directly to English
Useful if downstream NLP components (intent detection, etc.) only work in English:

In [ ]:
result_en = model.transcribe(audio_path, task="translate")
print("Original language:", result_en["language"])
print("English translation:", result_en["text"])

## 4. (Optional) Record audio live from the microphone

Useful for testing the assistant end-to-end without pre-recorded files. Only works when running the notebook locally with a microphone (won't work on a headless/remote server).

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write

def record_audio(filename="mic_input.wav", duration=5, samplerate=16000):
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype="int16")
    sd.wait()
    write(filename, samplerate, audio)
    print(f"Saved to {filename}")
    return filename

# recorded_file = record_audio(duration=5)
# result = model.transcribe(recorded_file)
# print(result["text"])

## 5. Reusable `SpeechToText` class

Wrap the model into a small class so the rest of your assistant (translation, intent detection, TTS reply, etc.) can just call `.transcribe(path)` without knowing about Whisper internals.

In [ ]:
class SpeechToText:
    def __init__(self, model_size="small", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = whisper.load_model(model_size, device=self.device)

    def transcribe(self, audio_path, language=None, translate_to_english=False):
        task = "translate" if translate_to_english else "transcribe"
        result = self.model.transcribe(audio_path, language=language, task=task)
        return {
            "text": result["text"].strip(),
            "language": result["language"],
        }


# Example usage:
# stt = SpeechToText(model_size="small")
# output = stt.transcribe("sample.wav")
# print(output["language"], "->", output["text"])

## Next steps for the assistant pipeline
1. Feed `SpeechToText().transcribe(audio)["text"]` into your NLP/intent-detection or LLM component.
2. Use the detected `language` to pick a matching text-to-speech voice for the reply.
3. If deploying as a web app, replace the mic-recording cell with a browser audio recorder (e.g. `streamlit-webrtc` or a JS `MediaRecorder` widget) that uploads the `.wav` file to this backend.